In [0]:
# ==============================================================================
# CAMADA SILVER: Limpeza, Tratamento de Qualidade e Padronização
# ==============================================================================

from pyspark.sql.functions import col, regexp_replace, when, lower, current_timestamp

# 1. Leitura da Tabela Delta da Camada Bronze
df_bronze = spark.table("workspace.default.bronze_listings")

print("Iniciando a transformação da Camada Bronze para Silver...")

# 2. Seleção de colunas relevantes, casting e tratamento de tipos
df_silver_step1 = df_bronze.select(
    col("id").cast("long").alias("listing_id"),
    col("host_id").cast("long"),
    col("host_name"),
    col("host_since").cast("date"),
    when(col("host_is_superhost") == "t", True).otherwise(False).alias("is_superhost"),
    col("host_listings_count").cast("integer"),
    col("neighbourhood_cleansed").alias("neighbourhood"),
    col("latitude").cast("double"),
    col("longitude").cast("double"),
    col("property_type"),
    col("room_type"),
    col("accommodates").cast("integer"),
    col("bedrooms").cast("integer"),
    col("beds").cast("integer"),
    # Limpeza financeira: remoção de cifrão e vírgulas para conversão numérico
    regexp_replace(regexp_replace(col("price"), "\\$", ""), ",", "").cast("decimal(10,2)").alias("price"),
    col("minimum_nights").cast("integer"),
    col("maximum_nights").cast("integer"),
    col("number_of_reviews").cast("integer"),
    col("review_scores_rating").cast("double").alias("review_score"),
    col("amenities")
)

# 3. Mapeamento de Zonas Geográficas do Rio de Janeiro
zona_sul = ["Copacabana", "Ipanema", "Leblon", "Leme", "Botafogo", "Flamengo", "Gávea", "Jardim Botânico", "Humaitá", "Catete", "Urca", "São Conrado", "Laranjeiras"]
zona_oeste = ["Barra da Tijuca", "Recreio dos Bandeirantes", "Jacarepaguá", "Campo Grande", "Bangu", "Itanhangá", "Vargem Grande"]
zona_norte = ["Tijuca", "Maracanã", "Vila Isabel", "Méier", "Madureira", "Penha", "Ilha do Governador", "Bonsucesso"]
centro = ["Centro", "Lapa", "Santa Teresa", "Glória", "Estácio"]

df_silver_step2 = df_silver_step1.withColumn(
    "zone",
    when(col("neighbourhood").isin(zona_sul), "Zona Sul")
    .when(col("neighbourhood").isin(zona_oeste), "Zona Oeste")
    .when(col("neighbourhood").isin(zona_norte), "Zona Norte")
    .when(col("neighbourhood").isin(centro), "Centro")
    .otherwise("Outros")
)

# 4. Extração de Comodidades Críticas (Atributos Qualitativos)
df_silver_clean = df_silver_step2 \
    .withColumn("has_wifi", lower(col("amenities")).contains("wifi")) \
    .withColumn("has_air_conditioning", lower(col("amenities")).contains("air conditioning")) \
    .withColumn("has_pool", lower(col("amenities")).contains("pool")) \
    .withColumn("has_sea_view", lower(col("amenities")).contains("view")) \
    .drop("amenities") \
    .withColumn("_processed_timestamp", current_timestamp())

# 5. Persistência na Tabela Delta Silver
target_table_silver = "workspace.default.silver_listings"

print(f"Persistindo dados limpos na tabela Delta: {target_table_silver}...")
df_silver_clean.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_table_silver)

print("✅ Camada Silver criada com sucesso! Exibindo amostra:")
display(spark.table(target_table_silver))